# Evidências de validação — robô de triagem de pedidos de serviços públicos

Este notebook reúne as evidências pedidas no enunciado (§5): execução com dados de teste, **resultado medido** em
uma amostra rotulada, **análise dos erros**, **casos de falha tratados** e **limites** da solução.

Ele usa os pacotes de `src/triage` (não reimplementa lógica) e roda do início ao fim com
`jupyter nbconvert --to notebook --execute notebooks/evidence.ipynb`. Todos os dados são sintéticos; nenhuma
chave de API é usada e o LLM fica **desligado** em todas as seções.

In [1]:
import dataclasses
import logging
import os
import shutil
import sys
from pathlib import Path
from unittest.mock import MagicMock, patch

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de data/ e do léxico de urgência são relativos à raiz do repositório

import run  # ponto de entrada `python run.py`
from triage.automation.allowlist import load_allowed_suffixes
from triage.automation.catalog import load_catalog
from triage.automation.link_check import check_link
from triage.automation.llm_client import OpenAICompatibleClient
from triage.automation.resolver import CatalogLookup, build_default_lookup
from triage.automation.response import build_response
from triage.classification import classify, triage_decision
from triage.classification.decision import ACTION_HUMAN_REVIEW
from triage.classification.model import ModelNotAvailableError, load, train
from triage.classification.preprocessing import normalize
from triage.config import load_settings
from triage.evaluation import (
    accuracy,
    automatic_accuracy,
    automatic_coverage,
    confusion_matrix,
    per_class_metrics,
    review_rate,
)
from triage.ingestion import Request
from triage.logging_setup import configure_logging, shutdown_logging
from triage.pipeline import Deps, process_request, run_batch
from triage.pipeline.report import render_markdown
from triage.storage import Repository

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

WORK_DIR = Path("output") / "notebook-evidence"  # temporário e ignorado pelo Git; removido na última célula
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

# Configuração base: lê o ambiente, mas sem `.env` e com o LLM sempre desligado.
BASE_SETTINGS = dataclasses.replace(
    load_settings(Path("sem.env")), llm_enabled=False, output_dir=str(WORK_DIR), log_level="INFO"
)
LABELED_CSV = Path("data/labeled/requests_labeled.csv")
print("Pasta de trabalho temporária:", WORK_DIR)

Pasta de trabalho temporária: output\notebook-evidence


## 1. Execução ponta a ponta

O robô lê todos os arquivos de `data/samples` (txt, csv e json, incluindo arquivos propositalmente defeituosos),
anonimiza, classifica, resolve o link do catálogo (verificado de verdade na rede) e grava o relatório.
O modelo é treinado aqui, em memória, só com `split=train` (o mesmo que `python scripts/train_model.py` faz).
Se a rede estiver indisponível, os links aparecem como "fora do ar" e os pedidos vão à revisão: é o caminho seguro.

In [2]:
model = train(LABELED_CSV)
e2e_settings = dataclasses.replace(BASE_SETTINGS, output_dir=str(WORK_DIR / "e2e"))
configure_logging(e2e_settings)
deps = Deps(
    repository=Repository(WORK_DIR / "e2e" / "triage.db"),
    lookup=build_default_lookup(),
    settings=e2e_settings,
    model=model,
    llm_client=None,
)
report = run_batch("data/samples", deps)
shutdown_logging()
print(render_markdown(report))

2026-11-10 11:57:22,340 WARNING triage.ingestion Pedido rejeitado: encoding_quebrado.txt (unreadable_encoding)


2026-11-10 11:57:22,342 WARNING triage.ingestion Pedido rejeitado: gigante.txt (too_large)


2026-11-10 11:57:22,343 WARNING triage.ingestion Pedido rejeitado: invalido.json (invalid_json)


2026-11-10 11:57:22,344 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#3 (text_too_short)


2026-11-10 11:57:22,344 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#4 (missing_text)


2026-11-10 11:57:22,345 WARNING triage.ingestion Pedido rejeitado: sem_coluna_text.csv (missing_text_column)


2026-11-10 11:57:22,345 WARNING triage.ingestion Pedido rejeitado: vazio.txt (empty_file)


2026-11-10 11:57:27,059 INFO triage.pipeline Relatório gravado em output\notebook-evidence\e2e\reports\run-20261110T145722-1.json


2026-11-10 11:57:27,060 INFO triage.pipeline Execução 1 concluída: {'answered': 4, 'queued_for_review': 4, 'rejected': 7, 'skipped_duplicate': 0, 'error': 0}


# Relatório da execução 1

- Início: 2026-11-10T14:57:22.267502+00:00
- Fim: 2026-11-10T14:57:27.049486+00:00
- Duração: 4.782 s

## Totais por status

| Item | Quantidade |
|---|---|
| respondidos | 4 |
| encaminhados à revisão | 4 |
| rejeitados na ingestão | 7 |
| já processados (ignorados) | 0 |
| com erro | 0 |

## Por categoria

| Item | Quantidade |
|---|---|
| birth_certificate | 1 |
| driver_license | 1 |
| id_card | 1 |
| other | 3 |
| social_security | 1 |
| voter_id | 1 |

## Por motivo de revisão

| Item | Quantidade |
|---|---|
| invalid_input | 7 |
| low_confidence | 1 |
| out_of_catalog | 2 |
| urgent | 1 |

## Links do catálogo (por pedido)

| Item | Quantidade |
|---|---|
| no ar | 4 |
| fora do ar | 0 |

## LLM

- Respostas reformuladas pelo LLM: 0
- Fallbacks para o template: nenhum



Rodar a mesma pasta de novo não refaz nada (idempotência): os pedidos já gravados são ignorados.

In [3]:
configure_logging(e2e_settings)
second = run_batch("data/samples", deps)
shutdown_logging()
print("Segunda execução, totais:", second.totals)

2026-11-10 11:57:27,081 WARNING triage.ingestion Pedido rejeitado: encoding_quebrado.txt (unreadable_encoding)


2026-11-10 11:57:27,084 WARNING triage.ingestion Pedido rejeitado: gigante.txt (too_large)


2026-11-10 11:57:27,085 WARNING triage.ingestion Pedido rejeitado: invalido.json (invalid_json)


2026-11-10 11:57:27,086 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#3 (text_too_short)


2026-11-10 11:57:27,086 WARNING triage.ingestion Pedido rejeitado: pedidos_bons.json#4 (missing_text)


2026-11-10 11:57:27,088 WARNING triage.ingestion Pedido rejeitado: sem_coluna_text.csv (missing_text_column)


2026-11-10 11:57:27,088 WARNING triage.ingestion Pedido rejeitado: vazio.txt (empty_file)


2026-11-10 11:57:27,105 INFO triage.pipeline Relatório gravado em output\notebook-evidence\e2e\reports\run-20261110T145727-2.json


2026-11-10 11:57:27,106 INFO triage.pipeline Execução 2 concluída: {'answered': 0, 'queued_for_review': 0, 'rejected': 0, 'skipped_duplicate': 15, 'error': 0}


Segunda execução, totais: {'answered': 0, 'queued_for_review': 0, 'rejected': 0, 'skipped_duplicate': 15, 'error': 0}


## 2. Métricas de classificação (conjunto de teste, sem LLM)

- **Conjunto:** `split=test` de `data/labeled/requests_labeled.csv`, gravado no próprio arquivo (6 pedidos por
  categoria, 9 categorias). O modelo foi treinado **só** com `split=train`: a célula abaixo confere que nenhum
  `id` de teste aparece no treino.
- **Sem LLM:** o LLM só reformula a resposta já decidida (ADR 005); classificar e decidir a revisão são
  determinísticos, então o acerto medido não depende dele.
- **Decisão:** `classify` + `triage_decision` com os limiares padrão (`CONFIDENCE_MIN=0.60`, `AMBIGUITY_MARGIN=0.15`).

In [4]:
labeled = pd.read_csv(LABELED_CSV, encoding="utf-8")
train_ids = set(labeled.loc[labeled["split"] == "train", "id"])
test = labeled[labeled["split"] == "test"].reset_index(drop=True)
assert not train_ids & set(test["id"]), "vazamento: pedido de teste no treino"


def triage_row(row):
    classification = classify(row.text, model)
    decision = triage_decision(row.text, classification, BASE_SETTINGS)
    return {
        "id": row.id,
        "text": row.text,
        "real": row.label,
        "previsto": classification.label,
        "fonte": classification.source,
        "confianca": round(decision.confidence, 2),
        "revisao": decision.action == ACTION_HUMAN_REVIEW,
        "motivo": decision.reason.value if decision.reason else "",
    }


results = pd.DataFrame([triage_row(row) for row in test.itertuples()])
labels = sorted(model.classes_)
y_true, y_pred, reviewed = results["real"].tolist(), results["previsto"].tolist(), results["revisao"].tolist()

print(f"Tamanho da amostra de teste: {len(results)} pedidos ({len(results) // len(labels)} por categoria)")
print(f"Acurácia geral: {accuracy(y_true, y_pred):.1%}  (um pedido a mais ou a menos muda {1 / len(results):.1%})")
print(f"Taxa de pedidos enviados à revisão humana: {review_rate(reviewed):.1%}")
print(f"Cobertura automática (respondidos sem revisão): {automatic_coverage(reviewed):.1%}")
answered = len(results) - sum(reviewed)
print(f"Acurácia entre os respondidos sozinhos: {automatic_accuracy(y_true, y_pred, reviewed):.1%}  ({answered} pedidos)")

Tamanho da amostra de teste: 54 pedidos (6 por categoria)
Acurácia geral: 81.5%  (um pedido a mais ou a menos muda 1.9%)
Taxa de pedidos enviados à revisão humana: 42.6%
Cobertura automática (respondidos sem revisão): 57.4%
Acurácia entre os respondidos sozinhos: 100.0%  (31 pedidos)


In [5]:
by_class = per_class_metrics(y_true, y_pred, labels)
table = pd.DataFrame(
    [
        {"classe": label, "precisão": m.precision, "recall": m.recall, "F1": m.f1, "pedidos (suporte)": m.support}
        for label, m in by_class.items()
    ]
).set_index("classe")
table.round(2)

,precisão,recall,F1,pedidos (suporte)
classe,,,,
birth_certificate,1.00,1.00,1.00,6
cpf_registration,0.86,1.00,0.92,6
driver_license,0.71,0.83,0.77,6
id_card,0.86,1.00,0.92,6
other,0.50,0.50,0.50,6
property_tax,0.75,1.00,0.86,6
social_benefits,0.83,0.83,0.83,6
social_security,1.00,0.67,0.80,6
voter_id,1.00,0.50,0.67,6


In [6]:
matrix = pd.DataFrame(confusion_matrix(y_true, y_pred, labels), index=labels, columns=labels)
matrix.index.name = "real ↓ / previsto →"
matrix

,birth_certificate,cpf_registration,driver_license,id_card,other,property_tax,social_benefits,social_security,voter_id
real ↓ / previsto →,,,,,,,,,
birth_certificate,6,0,0,0,0,0,0,0,0
cpf_registration,0,6,0,0,0,0,0,0,0
driver_license,0,1,5,0,0,0,0,0,0
id_card,0,0,0,6,0,0,0,0,0
other,0,0,0,1,3,2,0,0,0
property_tax,0,0,0,0,0,6,0,0,0
social_benefits,0,0,0,0,1,0,5,0,0
social_security,0,0,1,0,1,0,0,4,0
voter_id,0,0,1,0,1,0,1,0,3


In [7]:
print("Motivos de revisão entre os pedidos do teste:")
print(results.loc[results["revisao"], "motivo"].value_counts().to_string())
print()
print("Pedidos enviados à revisão, por categoria real:")
print(results.groupby("real")["revisao"].sum().astype(int).to_string())

Motivos de revisão entre os pedidos do teste:
motivo
low_confidence    16
out_of_catalog     6
urgent             1

Pedidos enviados à revisão, por categoria real:
real
birth_certificate    1
cpf_registration     0
driver_license       3
id_card              1
other                6
property_tax         3
social_benefits      3
social_security      3
voter_id             3


### Limiares

Os limiares da 02.02 **se mantiveram** (`CONFIDENCE_MIN=0.60`, `AMBIGUITY_MARGIN=0.15`). Motivo: nenhum dos pedidos
respondidos sozinhos foi errado nesta amostra, então não há erro automático a reduzir; e subir ou descer os
limiares olhando para este mesmo conjunto de teste o contaminaria (ajuste feito sobre o teste mede o ajuste, não o
modelo). Por isso não há tabela antes/depois.

## 3. Análise de erros

Todos os pedidos em que a categoria prevista difere da real. A coluna **fora do vocabulário** lista os termos do
pedido que o TF-IDF nunca viu no treino (medido, não suposto); **destino** diz o que o robô fez com o pedido.

In [8]:
vocabulary = set(model.named_steps["tfidf"].vocabulary_)


def unseen_terms(text):
    tokens = normalize(text).split()
    return ", ".join(sorted({token for token in tokens if token not in vocabulary}))


errors = results[results["real"] != results["previsto"]].copy()
errors["fora do vocabulário"] = errors["text"].map(unseen_terms)
errors["destino"] = errors["motivo"].map(lambda reason: f"revisão humana ({reason})" if reason else "respondido sozinho")
errors[["id", "text", "real", "previsto", "fonte", "confianca", "fora do vocabulário", "destino"]].set_index("id")

,text,real,previsto,fonte,confianca,fora do vocabulário,destino
id,,,,,,,
L017,"preciso do CPF para tirar a CNH, qual o procedimento?",driver_license,cpf_registration,model,0.57,procedimento,revisão humana (low_confidence)
L087,"Preciso justificar minha ausência na eleição, como faço?",voter_id,driver_license,model,0.20,"ausencia, eleicao, justificar",revisão humana (low_confidence)
L094,"não votei nas últimas eleições, tem multa? como pagar",voter_id,other,model,0.29,"eleicoes, pagar, ultimas, votei",revisão humana (out_of_catalog)
L100,"mesário voluntário, como me inscrever",voter_id,social_benefits,model,0.16,"mesario, voluntario",revisão humana (low_confidence)
L117,"Preciso do NIS, como descobrir meu número?",social_benefits,other,model,0.15,"descobrir, nis, numero",revisão humana (out_of_catalog)
L130,como emitir o meu CNIS,social_security,other,model,0.33,cnis,revisão humana (out_of_catalog)
L140,como fazer a prova de vida do aposentado,social_security,driver_license,model,0.18,vida,revisão humana (low_confidence)
L164,"preciso renovar meu passaporte, perdi o prazo",other,property_tax,model,0.18,,revisão humana (low_confidence)
L167,"buraco enorme na rua da minha casa, quem conserta?",other,property_tax,model,0.31,"buraco, conserta, enorme, rua",revisão humana (low_confidence)


In [9]:
print(f"Erros de classificação: {len(errors)} de {len(results)} pedidos ({len(errors) / len(results):.1%})")
print(f"Erros que o robô respondeu sozinho (sem revisão): {int((~errors['revisao']).sum())}")
print(f"Erros encaminhados à revisão humana: {int(errors['revisao'].sum())}")

Erros de classificação: 10 de 54 pedidos (18.5%)
Erros que o robô respondeu sozinho (sem revisão): 0
Erros encaminhados à revisão humana: 10


**Agrupamento e causa provável** (interpretação a partir da tabela acima; a causa é uma hipótese, não foi testada
com experimento próprio):

| Grupo | Pedidos | Causa provável |
|---|---|---|
| Assunto que o treino cobre pouco | L087, L094, L100 (título de eleitor); L117 (NIS); L130, L140 (aposentadoria) | O pedido usa termos específicos (justificativa de ausência, mesário, NIS, CNIS, prova de vida), ausentes ou raros no treino; com poucos termos conhecidos, a probabilidade se espalha e a confiança fica baixa. |
| Duas categorias no mesmo pedido | L017 (CPF para tirar a CNH) | O texto cita CPF e CNH; duas regras casam e a decisão cai no modelo, que escolheu a outra categoria com confiança 0,57. É a ambiguidade que o conjunto traz de propósito. |
| Categoria `other` heterogênea | L164 (passaporte), L167 (buraco na rua), L177 (reclamação do posto de saúde) | `other` mistura assuntos sem relação entre si; palavras soltas ("casa", "prazo", "telefone") aproximam o pedido de um serviço do catálogo. |

Em todos os casos a confiança ficou abaixo do limite ou a categoria prevista foi `other`, e por isso o pedido foi
para a revisão humana em vez de receber um link errado.

## 4. Casos de falha tratados

Cada caso provoca a falha de verdade (ou a simula, quando indicado), mostra o **log** gerado (saída de erro da
célula) e o **caminho seguro** que o robô seguiu. A execução nunca é interrompida e nenhum log carrega o texto do
cidadão.

In [10]:
def case_settings(name):
    folder = WORK_DIR / name
    settings = dataclasses.replace(BASE_SETTINGS, output_dir=str(folder))
    configure_logging(settings)
    return settings, folder


def fresh_deps(settings, folder, **overrides):
    values = {
        "repository": Repository(folder / "triage.db"),
        "lookup": build_default_lookup(),
        "settings": settings,
        "model": model,
        "llm_client": None,
    }
    return Deps(**{**values, **overrides})


CLEAR_REQUEST = Request("DEMO-1", "Como faço para renovar a minha CNH?", "notebook")

### 4.1 Arquivo ausente (pasta de entrada inexistente)

In [11]:
settings, folder = case_settings("ausente")
exit_code = run.main(["--input", "pasta_que_nao_existe", "--output", str(folder), "--no-llm"])
print("Código de saída:", exit_code, "(2 = entrada inválida; a execução terminou sem exceção)")

2026-11-10 11:57:27,276 ERROR triage.run Pasta de entrada não encontrada: pasta_que_nao_existe


Código de saída: 2 (2 = entrada inválida; a execução terminou sem exceção)


### 4.2 JSON inválido

`data/samples/invalido.json` tem o JSON truncado. O arquivo é rejeitado, registrado e enviado à fila de revisão com
o motivo `invalid_input`; os demais arquivos da pasta seguem normalmente.

In [12]:
settings, folder = case_settings("json")
input_dir = folder / "entrada"
input_dir.mkdir()
shutil.copy("data/samples/invalido.json", input_dir)
shutil.copy("data/samples/pedido_bom.txt", input_dir)
json_report = run_batch(input_dir, fresh_deps(settings, folder))
shutdown_logging()
print("Totais:", json_report.totals)
print("Motivos de revisão:", json_report.by_review_reason)

2026-11-10 11:57:27,331 WARNING triage.ingestion Pedido rejeitado: invalido.json (invalid_json)


2026-11-10 11:57:28,524 INFO triage.pipeline Relatório gravado em output\notebook-evidence\json\reports\run-20261110T145727-1.json


2026-11-10 11:57:28,525 INFO triage.pipeline Execução 1 concluída: {'answered': 1, 'queued_for_review': 0, 'rejected': 1, 'skipped_duplicate': 0, 'error': 0}


Totais: {'answered': 1, 'queued_for_review': 0, 'rejected': 1, 'skipped_duplicate': 0, 'error': 0}
Motivos de revisão: {'invalid_input': 1}


### 4.3 Link fora do ar (simulado com uma porta local fechada)

O verificador de link real (`check_link`, com timeout e retry com espera) aponta para uma porta local sem serviço.
Ele tenta de novo, desiste sem levantar exceção e o pedido, que seria respondido, vai à revisão humana
(`link_unavailable`): o robô não entrega link que não conseguiu confirmar.

In [13]:
def link_always_down(_url):
    return check_link("http://127.0.0.1:9/", timeout=0.5, retries=2, backoff=0.1)


settings, folder = case_settings("link")
down_lookup = CatalogLookup(load_catalog(), load_allowed_suffixes(), link_always_down)
result = process_request(CLEAR_REQUEST, fresh_deps(settings, folder, lookup=down_lookup))
shutdown_logging()
print("Desfecho:", result.status, "| categoria:", result.category, "| motivo:", result.reason, "| link:", result.link_status)

2026-11-10 11:57:29,070 WARNING triage.retry Tentativa 1/2 falhou (HTTPConnectionPool(host='127.0.0.1', port=9): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='127.0.0.1', port=9) at 0x1c63ffcab10>, 'Connection to 127.0.0.1 timed out. (connect timeout=0.5)'))); nova tentativa em 0.10s


2026-11-10 11:57:29,681 ERROR triage.retry Tentativa 2/2 falhou; desistindo: HTTPConnectionPool(host='127.0.0.1', port=9): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<HTTPConnection(host='127.0.0.1', port=9) at 0x1c63ffd5990>, 'Connection to 127.0.0.1 timed out. (connect timeout=0.5)'))


2026-11-10 11:57:29,683 WARNING triage.automation Link inacessível (ConnectTimeout)


2026-11-10 11:57:29,684 WARNING triage.automation Link de driver_license indisponível (ConnectTimeout)


Desfecho: queued_for_review | categoria: driver_license | motivo: link_unavailable | link: unavailable


### 4.4 Resposta vazia do LLM (simulada)

A chamada HTTP é substituída por uma resposta 200 com `content` vazio, que o cliente trata como resposta inválida.
A resposta final cai no template do catálogo e o motivo fica registrado. Nenhuma chamada real é feita.

In [14]:
empty_reply = MagicMock(status_code=200)
empty_reply.json.return_value = {"choices": [{"message": {"content": ""}}]}
settings, folder = case_settings("llm")
service = load_catalog()["driver_license"]
client = OpenAICompatibleClient("http://llm.invalid/v1", "modelo-de-teste", "valor-ficticio")
with patch("triage.automation.llm_client.requests.post", return_value=empty_reply):
    response = build_response(service, client)
shutdown_logging()
print("Usou o LLM:", response.used_llm, "| motivo do fallback:", response.fallback_reason)
print("Resposta entregue (template):", response.text)

2026-11-10 11:57:29,734 WARNING triage.automation Resposta inválida do LLM (conteúdo vazio ou não textual); usando o template


Usou o LLM: False | motivo do fallback: llm_invalid_response
Resposta entregue (template): Para resolver o seu pedido, o serviço indicado é: Carteira Nacional de Habilitação (CNH).
Órgão responsável: Secretaria Nacional de Trânsito (Senatran) e Detrans estaduais.
A página do gov.br explica como emitir a CNH digital. Primeira habilitação, renovação e segunda via são atendidas pelo Detran do seu estado, com login gov.br.
Acesse o link oficial: https://www.gov.br/pt-br/servicos/emitir-a-carteira-nacional-de-habilitacao-digital-cnh-e


### 4.5 Modelo ausente

O arquivo do modelo não existe: `load` levanta `ModelNotAvailableError`, o robô segue **sem modelo** e todo pedido
vai à revisão humana, em vez de a execução parar.

In [15]:
settings, folder = case_settings("modelo")
try:
    load(folder / "nao_existe.joblib")
except ModelNotAvailableError as error:
    logging.getLogger("triage.pipeline").warning(
        "Modelo indisponível (%s); todos os pedidos irão à revisão humana", type(error).__name__
    )
result = process_request(CLEAR_REQUEST, fresh_deps(settings, folder, model=None))
shutdown_logging()
print("Desfecho:", result.status, "| motivo:", result.reason)

2026-11-10 11:57:29,767 WARNING triage.pipeline Modelo indisponível (ModelNotAvailableError); todos os pedidos irão à revisão humana


Desfecho: queued_for_review | motivo: low_confidence


## 5. Limites da solução

Escrito só com o que foi medido ou é fato do desenho:

- **Amostra pequena e sintética.** O teste tem 54 pedidos (6 por categoria), escritos à mão para este protótipo.
  Cada pedido vale cerca de 1,9 ponto percentual na acurácia, e 6 pedidos por classe não sustentam conclusão
  estatística por categoria. Os números mostram o comportamento nesta amostra, não o desempenho com pedidos reais.
- **Sem dados reais.** Não há nomes, endereços nem outros identificadores além de CPF, telefone e e-mail; esses
  três são anonimizados por regex e o resto do texto não é tratado. Nome e endereço escritos no pedido passariam.
- **Catálogo estático.** São 8 serviços curados à mão; o robô não busca informação ao vivo e o link só vale até o
  órgão mudar a página (por isso o link é verificado a cada execução).
- **Revisão humana alta.** Uma parte grande dos pedidos do teste vai à revisão (seção 2): o robô prefere pedir
  ajuda humana a arriscar um link errado, e isso custa cobertura automática.
- **Ambiguidade.** Pedido com dois assuntos (como CPF e CNH) cai na dúvida e vai à revisão; o robô não pergunta de volta.
- **Sem multi-turno.** Cada pedido é independente; não há conversa nem pedido de esclarecimento.
- **LLM opcional e não medido.** Só reformula a resposta; ficou desligado nas medições e a chamada real a um provedor
  não foi exercitada aqui, só os caminhos de falha simulados.
- **Sem autenticação.** Quem tem acesso à máquina vê a fila de revisão (ADR 003).

In [16]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
if Path("output").is_dir() and not any(Path("output").iterdir()):
    Path("output").rmdir()
print("Pasta temporária removida.")

Pasta temporária removida.
